In [4]:
!pip install -qU --force-reinstall langchain-core>=1.4.0 langchain-community langchain-mistralai faiss-cpu mistralai pandas==2.2.2 requests==2.32.4 opentelemetry-api==1.38.0 langchain-text-splitters>=1.1.2 opentelemetry-semantic-conventions==0.59b0
# After running this cell, please restart your Colab runtime (Runtime -> Restart runtime) for the changes to take effect.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
gradio-client 1.14.0 requires websockets<16.0,>=13.0, but you have websockets 16.0 which is incompatible.
langgraph 1.2.1 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
g

In [1]:
import pandas as pd
import os
import json
from google.colab import userdata
from langchain_community.document_loaders import DataFrameLoader
from langchain_mistralai import MistralAIEmbeddings, ChatMistralAI
from langchain_community.vectorstores import FAISS
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage
from langchain_core.tools import create_retriever_tool

# ========================================================
# 1. CARGA Y PREPARACIÓN DEL DATASET
# ========================================================
print("⏳ Cargando y normalizando el archivo CSV...")
df = pd.read_csv('sales_data_sample.csv', encoding='latin1')

# Limpieza básica
df = df.drop(columns=['ADDRESSLINE2', 'PHONE', 'ADDRESSLINE1', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME'], errors='ignore')
df['STATE'] = df['STATE'].fillna('N/A')
df['TERRITORY'] = df['TERRITORY'].fillna('Desconocido')
df['POSTALCODE'] = df['POSTALCODE'].fillna('00000')
df = df.drop_duplicates()

# Normalización a minúsculas
columnas_texto = ['STATUS', 'PRODUCTLINE', 'CUSTOMERNAME', 'CITY', 'COUNTRY', 'DEALSIZE']
for col in columnas_texto:
    df[col] = df[col].astype(str).str.lower().str.strip()

df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'], errors='coerce').dt.strftime('%Y-%m-%d')

# Crear columna de contexto en español para el RAG
df['TEXT_CONTEXT_ES'] = df.apply(lambda row:
    f"El cliente {row['CUSTOMERNAME']} de la ciudad de {row['CITY']}, {row['COUNTRY']}, "
    f"compró el producto con código {row['PRODUCTCODE']} (Línea: {row['PRODUCTLINE']}) "
    f"el día {row['ORDERDATE']}. Se ordenaron {row['QUANTITYORDERED']} unidades a un precio de "
    f"{row['PRICEEACH']} cada una, sumando un total de {row['SALES']} USD en ventas. "
    f"El estado actual del pedido es {row['STATUS']} y el tamaño del trato se clasifica como {row['DEALSIZE']}.",
    axis=1
)
print("✅ Dataset preparado.")

# ========================================================
# 2. CREACIÓN DEL CORPUS VECTORIAL (RAG CON FAISS)
# ========================================================
print("⏳ Indexando los textos en la base vectorial FAISS...")
os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')

loader = DataFrameLoader(df, page_content_column="TEXT_CONTEXT_ES")
documentos = loader.load()

# Usamos FAISS que es ultra estable con tu nueva configuración
embeddings = MistralAIEmbeddings(model="mistral-embed")
vector_store = FAISS.from_documents(documentos, embeddings)
print("✅ RAG configurado exitosamente.")

# ========================================================
# 3. CONFIGURACIÓN DEL AGENTE MODERNO DE LANGCHAIN
# ========================================================
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
tool_rag = create_retriever_tool(
    retriever,
    "buscar_en_ventas",
    "Úsala para buscar información detallada sobre las ventas, clientes, productos, pedidos, países y estados de cuenta."
)

# Inicializamos Mistral y vinculamos la herramienta directamente
llm = ChatMistralAI(model="mistral-small-latest", temperature=0)
llm_con_herramientas = llm.bind_tools([tool_rag])

def ejecutar_agente_langchain(pregunta_usuario: str):
    print(f"\n💬 **Pregunta:** {pregunta_usuario}")

    messages = [
        SystemMessage(content="Eres un asistente experto en análisis de datos de ventas de la empresa. "
                             "Responde siempre en español de forma clara y concisa basándote estrictamente en la herramienta 'buscar_en_ventas'."),
        HumanMessage(content=pregunta_usuario)
    ]

    respuesta_modelo = llm_con_herramientas.invoke(messages)

    if respuesta_modelo.tool_calls:
        print("🤖 [Agente LangChain]: Buscando en la base de datos vectorial...")
        messages.append(respuesta_modelo)

        for tool_call in respuesta_modelo.tool_calls:
            resultado_txt = tool_rag.invoke(tool_call['args'])
            messages.append(ToolMessage(content=resultado_txt, tool_call_id=tool_call['id']))

        print("✍️ [Mistral]: Redactando respuesta con los datos encontrados...")
        respuesta_final = llm_con_herramientas.invoke(messages)
        return respuesta_final.content
    else:
        return respuesta_modelo.content

print("\n🚀 ¡TODO LISTO! El sistema unificado está activo y esperando consultas.")

⏳ Cargando y normalizando el archivo CSV...
✅ Dataset preparado.
⏳ Indexando los textos en la base vectorial FAISS...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

✅ RAG configurado exitosamente.

🚀 ¡TODO LISTO! El sistema unificado está activo y esperando consultas.


In [2]:
pregunta = "¿Qué me puedes decir sobre las ventas en la ciudad de nyc?"
resultado = ejecutar_agente_langchain(pregunta)

print("\n--- RESPUESTA FINAL ---")
print(resultado)


💬 **Pregunta:** ¿Qué me puedes decir sobre las ventas en la ciudad de nyc?
🤖 [Agente LangChain]: Buscando en la base de datos vectorial...
✍️ [Mistral]: Redactando respuesta con los datos encontrados...

--- RESPUESTA FINAL ---
En la ciudad de **NYC**, se registraron las siguientes ventas:

1. **Cliente:** Vitachrome Inc.
   - **Producto:** Código `S32_1268` (Línea: *Trucks and Buses*).
   - **Fecha:** 2004-11-05.
   - **Cantidad:** 20 unidades.
   - **Precio unitario:** 98.18 USD.
   - **Total:** 1,963.60 USD.
   - **Estado del pedido:** *Shipped*.
   - **Tamaño del trato:** *Small*.

2. **Cliente:** Vitachrome Inc.
   - **Producto:** Código `S32_2206` (Línea: *Motorcycles*).
   - **Fecha:** 2004-04-05.
   - **Cantidad:** 26 unidades.
   - **Precio unitario:** 40.23 USD.
   - **Total:** 1,045.98 USD.
   - **Estado del pedido:** *Shipped*.
   - **Tamaño del trato:** *Small*.

3. **Cliente:** Vitachrome Inc.
   - **Producto:** Código `S32_4485` (Línea: *Motorcycles*).
   - **Fecha:** 2

In [6]:
# Lista de preguntas para probar
preguntas = [
    "¿Qué compró 'land of toys inc.'?",
    "¿Qué ventas hay en 'nyc'?",
    "¿Cuál es el clima en Londres?"
]

# Bucle para probarlas una por una
for q in preguntas:
    print(ejecutar_agente_langchain(q))
    print("-" * 50)


💬 **Pregunta:** ¿Qué compró 'land of toys inc.'?
🤖 [Agente LangChain]: Buscando en la base de datos vectorial...
✍️ [Mistral]: Redactando respuesta con los datos encontrados...
**Land of Toys Inc.** realizó las siguientes compras:

1. **Producto:** S72_1253 (Línea: *planes*)
   - **Fecha:** 15 de noviembre de 2004
   - **Cantidad:** 44 unidades
   - **Precio unitario:** 86.13 USD
   - **Total:** 3,789.72 USD
   - **Estado del pedido:** *Shipped*
   - **Tamaño del trato:** *Medium*

2. **Producto:** S72_3212 (Línea: *ships*)
   - **Fecha:** 7 de mayo de 2004
   - **Cantidad:** 23 unidades
   - **Precio unitario:** 65.52 USD
   - **Total:** 1,506.96 USD
   - **Estado del pedido:** *Cancelled*
   - **Tamaño del trato:** *Small*

3. **Producto:** S24_3816 (Línea: *vintage cars*)
   - **Fecha:** 7 de mayo de 2004
   - **Cantidad:** 23 unidades
   - **Precio unitario:** 76.31 USD
   - **Total:** 1,755.13 USD
   - **Estado del pedido:** *Cancelled*
   - **Tamaño del trato:** *Small*

4. **Pr